In [ ]:
import argparse
import sys
import time

import openmm as mm
from openmm import unit

from topo import engine
from topo.utils.config import read_simulation_config
from topo.mdrun.protocol import run_protocol, describe_schedule

In [ ]:
cfg = read_simulation_config('md.ini')
cfg.prepare_output_dir()

In [ ]:
# === Step 1: build the coarse-grained system =============================
# Returns a BuiltSystem with .cgModel, .system, .topology, .positions.
built = engine.build_system(cfg)

In [ ]:
built

In [ ]:
# === Step 2: set up the OpenMM Simulation ===============================
# Integrator, platform/device, starting coordinates & velocities, restart.
ctx = engine.setup_simulation(cfg, built, control_file='md.ini')
sim = ctx.simulation

In [ ]:
# === Step 3: attach reporters (DCD/log/checkpoint) ======================
engine.attach_reporters(cfg, sim, suffix='', append=ctx.restart_active,
                        total_steps=cfg.md_steps)

In [ ]:
# === EDIT: define your temperature protocol =============================
# A schedule is just a list of (temperature, n_steps) stages. The default
# below is a single constant-ref_t production run -- identical to a plain
# `topo-mdrun` equilibrium run. Replace it with your own stages, e.g. a
# step-cooling ramp:
#
#   schedule = [(500 * unit.kelvin, 200000),
#               (400 * unit.kelvin, 200000),
#               (cfg.ref_t,         cfg.md_steps)]
schedule = [(cfg.ref_t, cfg.md_steps)]
print(f"Temperature protocol: {describe_schedule(schedule)}")

print('Simulation started')
start_time = time.time()

In [ ]:
# === Step 4: run ========================================================
# Simplest form -- run the whole schedule in one call (handles restart):
run_protocol(sim, schedule, done_steps=ctx.done_steps)

In [ ]:
# === Step 5: finalize ===================================================
# Save the final checkpoint + conformation, close out run metadata.
engine.finalize_simulation(cfg, ctx, built.topology, start_time)